In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

In [2]:
# Get sequence of EGFR
with open(DATA_RAW / "UNIPROT_P00533.FASTA") as f:
    lines = f.read().splitlines()

header = lines[0]
seq = "".join(lines[1:])
len(seq)

# exclude peptide, only chain
seq = seq[24:] 

### Kinase domain
The full EGFR (human) sequence is given by Uniprot ID P00533.
However, the full 1210 residue protein is very large and only the kinase domain is relevant.
Here the kinase domain used in the crystal structure of the erlotinib complex [672–998] is used. Some key residues for checking alignment include:  
- Tyr845  
- Glu848  
- Lys836  

ref: Stamos et al. 2002 Structure of the Epidermal Growth Factor Receptor Kinase Domain
Alone and in Complex with a 4-Anilinoquinazoline Inhibitor

In [3]:
start, final = 672, 998

kinase_seq = seq[start-1:final]

In [4]:
kinase_seq

'GEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQG'

In [5]:
import numpy as np
tyr = 845
glu = 848
lys = 836

# account for shift in indexing
print(f"Main chain residues: Tyr845 : {seq[tyr-1]}, Glu848 :{seq[glu-1]}, Lys836 : {seq[lys-1]}")
print(f"Kinase domain : Tyr845 : {kinase_seq[tyr-start]}, Glu848 :{kinase_seq[glu-start]}, Lys836 : {kinase_seq[lys-start]}")

Main chain residues: Tyr845 : Y, Glu848 :E, Lys836 : K
Kinase domain : Tyr845 : Y, Glu848 :E, Lys836 : K


In [6]:
import pandas as pd 

boltz_set = pd.read_csv(DATA_PROCESSED / "boltz_subset.csv")

In [7]:
# Make 1 yaml file per compound
import yaml

msa = ""

def write_yaml(smi, seq, name, msa=msa):
    protein = {"id": "A", "sequence": seq}
    if msa is not None:
        protein["msa"] = str(msa)

    spec = {
        "version": 1,
        "sequences": [
            {"protein": protein},
            {"ligand": {"id": "B", "smiles": smi}},
        ],
        "properties": [{"affinity": {"binder": "B"}}],
    }

    path = PROJECT_ROOT / "boltz_inputs" / f"{name}.yaml"
    with open(path, "w") as f:
        yaml.safe_dump(spec, f, sort_keys=False, default_flow_style=False)
    return path

In [8]:
# write yaml for CHEMBL104
chembl104 = boltz_set.loc[boltz_set["Molecule ChEMBL ID"] == "CHEMBL104"].iloc[0]

write_yaml(chembl104["Clean Smiles"], kinase_seq, chembl104["Molecule ChEMBL ID"])

WindowsPath('c:/Users/lauren/Documents/python/boltz-mini-project/boltz_inputs/CHEMBL104.yaml')

In an initial `boltz` run on CHEMBL104, the flag --use_msa_server was set. 
To avoid repeated computations of MSA on an identical sequence, the *.a3m file was added to data/raw/ and used for subsequent boltz computations. Command from `PROJECT_ROOT`:  

`boltz predict .\boltz_inputs\CHEMBL104.yaml --use_msa_server --out_dir .\boltz_outputs\ --no_kernels`

In [9]:
# write 51 yamls
for _, row in boltz_set.iterrows():
    if row["Molecule ChEMBL ID"] != "CHEMBL104":
        write_yaml(row["Clean Smiles"], kinase_seq, row["Molecule ChEMBL ID"], msa=DATA_RAW / "uniref.a3m")

In [10]:
boltz_set["Decoy"].sum()

10

In [11]:
boltz_set["Known Drug"].sum()

6

In [12]:
# Last sanity check on input SMILES before batch run
from rdkit.Chem import Draw, PandasTools

PandasTools.AddMoleculeColumnToFrame(boltz_set, smilesCol="Clean Smiles", 
                                        molCol="RDkit Mol", includeFingerprints=True)
PER_PAGE = 6

legends = [f"Potency: {row['Potency']}\nDecoy: {row['Decoy']}\nKnown Drug: {row['Known Drug']}" for _, row in boltz_set.iterrows()]

for page, start in enumerate(range(0, len(boltz_set["RDkit Mol"]), PER_PAGE), start=1):
    chunk_mols = boltz_set["RDkit Mol"][start:start + PER_PAGE]
    chunk_legends = legends[start:start + PER_PAGE]

    img = Draw.MolsToGridImage(chunk_mols, molsPerRow=3, subImgSize=(400, 400),
        legends=chunk_legends,returnPNG=False,)
    img.save(FIGURES / "boltz_subset_mols" / f"boltz_subset_page{page:02d}.png")